## **FIASS**

Facebook AI Similarity Search (FIASS) is a library for efficient similarity search and clustering of dense vecrors.

It contains algorithms that search in sets of vectors of ant size, up to ones that possibly do not fit in RAM. it also contains supporting code for evaluation and parameter tuning.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("Data/speech.txt")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

In [2]:
docs

[Document(metadata={'source': 'Data/speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦'),
 Document(metadata={'source': 'Data/speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us 

In [3]:
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(docs, embeddings)
db

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6578.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
query = "How does the speaker describe the desired outcome of the war?"
docs = db.similarity_search(query=query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

### Retriever

We can also convert the vectorstores into a RetrieverClass. This allows us to easily use it in other Langchain methods, which largely work with retrievers.

In [7]:
retriever = db.as_retriever()
doc = retriever.invoke(query)
doc[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

### Similarity search with Score

There are some FAISS specific methods. One of them is ```similarity_search_with_score()``` which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [8]:
docs_and_ss = db.similarity_search_with_score(query)
docs_and_ss[0]

(Document(id='d16442a3-0a65-49b6-8418-d8d78ec8b793', metadata={'source': 'Data/speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 np.float32(1.3070922))

In [9]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[-0.04968958720564842,
 0.10655161738395691,
 0.0015431185020133853,
 -0.013194341212511063,
 -0.044387806206941605,
 0.05857324227690697,
 0.04666466638445854,
 -0.024530693888664246,
 0.004829424899071455,
 0.020339960232377052,
 -0.03410932049155235,
 0.03873789682984352,
 0.016618920490145683,
 -0.018834849819540977,
 -0.007143084891140461,
 0.006980123929679394,
 0.020850062370300293,
 -0.03520536050200462,
 0.004866400733590126,
 -0.0072698090225458145,
 0.07890864461660385,
 0.056077100336551666,
 0.06984831392765045,
 0.016556687653064728,
 0.013394543901085854,
 -0.012157347984611988,
 -0.020828330889344215,
 0.05277489498257637,
 -0.020383475348353386,
 0.026147590950131416,
 0.04590854048728943,
 -0.017534684389829636,
 0.016683505848050117,
 0.0021405683364719152,
 0.006522961426526308,
 0.0013917300384491682,
 -0.054171763360500336,
 -0.016581425443291664,
 -0.011631510220468044,
 -0.020481334999203682,
 -0.0529186986386776,
 -0.03948397934436798,
 0.016547992825508118,
 0

In [10]:
docs_vec = db.similarity_search_by_vector(embedding_vector)
docs_vec[0]

Document(id='d16442a3-0a65-49b6-8418-d8d78ec8b793', metadata={'source': 'Data/speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.')

In [11]:
### Saving and Loading
db.save_local("FAISS_index")

In [12]:
## Load the folder
new_db = FAISS.load_local("FAISS_index", embeddings, allow_dangerous_deserialization=True)

In [13]:
docs = new_db.similarity_search(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'